# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [3]:
import pandas as pd

# Load the ranked baseline queue from Week 4
queue = pd.read_csv("../outputs/baseline_action_score.csv")

# Rank highest-priority pages first
queue = queue.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).copy()

# Add a priority rank
queue["priority_rank"] = range(1, len(queue) + 1)


# Turn scores/reason codes into human-readable actions
def assign_action(row):
    if row["reason_code"] == "weak_ctr_visible_stale":
        return "Review page for possible refresh"
    elif row["reason_code"] == "weak_ctr_visible":
        return "Review CTR and search context"
    else:
        return "Monitor — no immediate action"


queue["action"] = queue.apply(assign_action, axis=1)


# Every recommendation requires human review
queue["human_review_required"] = True


# Show the main playbook columns
playbook = queue[
    [
        "priority_rank",
        "content_id",
        "action",
        "reason_code",
        "score",
        "impressions_90d",
        "ctr",
        "avg_position",
        "days_since_last_update",
        "human_review_required"
    ]
]

playbook.head(20)

,priority_rank,content_id,action,reason_code,score,impressions_90d,ctr,avg_position,days_since_last_update,human_review_required
16751,1,content_cf56e2e2e282,Review page for possible refresh,weak_ctr_visible_stale,2,61678,0.15,19.7,194,True
21268,2,content_0a91db491d14,Review page for possible refresh,weak_ctr_visible_stale,2,13299,0.49,10.5,193,True
12045,3,content_c2d929d83eaa,Review page for possible refresh,weak_ctr_visible_stale,2,7558,0.20,17.9,193,True
5327,4,content_fe16a55cd13d,Review page for possible refresh,weak_ctr_visible_stale,2,4556,0.33,16.4,194,True
20837,5,content_928af3e22c80,Review page for possible refresh,weak_ctr_visible_stale,2,1697,0.12,15.8,193,True
22872,6,content_e3ff1b093148,Review page for possible refresh,weak_ctr_visible_stale,2,1408,0.28,7.8,183,True
26840,7,content_7f116ae1f6f5,Review page for possible refresh,weak_ctr_visible_stale,2,954,0.42,9.0,301,True
26799,8,content_77d4d5930e5e,Review page for possible refresh,weak_ctr_visible_stale,2,828,0.24,18.6,194,True
7452,9,content_72496874f806,Review page for possible refresh,weak_ctr_visible_stale,2,821,0.24,5.8,301,True
11630,10,content_6226ee6adc91,Review page for possible refresh,weak_ctr_visible_stale,2,545,0.18,17.8,183,True


In [5]:
# Show the top 20 recommendations from the playbook

playbook.head(20)

,priority_rank,content_id,action,reason_code,score,impressions_90d,ctr,avg_position,days_since_last_update,human_review_required
16751,1,content_cf56e2e2e282,Review page for possible refresh,weak_ctr_visible_stale,2,61678,0.15,19.7,194,True
21268,2,content_0a91db491d14,Review page for possible refresh,weak_ctr_visible_stale,2,13299,0.49,10.5,193,True
12045,3,content_c2d929d83eaa,Review page for possible refresh,weak_ctr_visible_stale,2,7558,0.20,17.9,193,True
5327,4,content_fe16a55cd13d,Review page for possible refresh,weak_ctr_visible_stale,2,4556,0.33,16.4,194,True
20837,5,content_928af3e22c80,Review page for possible refresh,weak_ctr_visible_stale,2,1697,0.12,15.8,193,True
22872,6,content_e3ff1b093148,Review page for possible refresh,weak_ctr_visible_stale,2,1408,0.28,7.8,183,True
26840,7,content_7f116ae1f6f5,Review page for possible refresh,weak_ctr_visible_stale,2,954,0.42,9.0,301,True
26799,8,content_77d4d5930e5e,Review page for possible refresh,weak_ctr_visible_stale,2,828,0.24,18.6,194,True
7452,9,content_72496874f806,Review page for possible refresh,weak_ctr_visible_stale,2,821,0.24,5.8,301,True
11630,10,content_6226ee6adc91,Review page for possible refresh,weak_ctr_visible_stale,2,545,0.18,17.8,183,True


### How to read the ranked action queue

The queue ranks content pages for human review rather than automatically recommending changes.

Pages with the reason code `weak_ctr_visible_stale` are given higher priority because they combine visible search exposure, relatively weak CTR, and a longer time since the last update. These pages should be reviewed for possible refresh.

Pages with the reason code `weak_ctr_visible` are also worth reviewing, but the first step is to check their search context. A low CTR may be influenced by query intent, SERP features, competition, or the type of page.

The priority rank is intended to help reviewers decide which pages to investigate first. It does not mean that a page is definitely underperforming or that updating it will improve its results. Every recommendation requires human review before action is taken.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This action playbook is intended to help content or SEO reviewers prioritize which anonymized content pages may be worth investigating first.

The ranked queue is decision-support, not an automated decision system. It uses observed search and content signals to highlight pages that may deserve review, particularly pages with visible search exposure and relatively weak CTR.

The recommendations are intended to save review time by providing an ordered starting point. They do not determine that a page is bad or guarantee that changing the page will improve performance.

### Limits

The recommendations are based on patterns in this dataset and should be interpreted directionally.

The playbook does not include the full context behind each page, such as the actual search queries, SERP features, competitor context, or business goals.

A low CTR may be normal for some types of queries or search-result environments. Similarly, a page being stale does not automatically mean it needs to be refreshed.

For these reasons, the ranked queue should guide human investigation rather than automatically trigger content updates.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review

Before acting on a recommendation, a human reviewer should check:

- Whether the page's CTR is actually low for its search queries and average position.
- The search intent and whether the page matches that intent.
- Available SERP context that may affect clicks.
- Whether the content is still accurate and useful.
- Whether a refresh is appropriate for the page's purpose and business goals.

The ranked queue provides a starting point for investigation. A recommendation should not automatically result in a content change.

### No-go list: what should NOT be automated

This playbook should not automatically:

- Rewrite or refresh a page without human review.
- Delete, merge, or prune content.
- Assume that low CTR means poor content.
- Guarantee that updating a page will improve clicks, rankings, or engagement.
- Make decisions based only on the model score or baseline score.

Human judgment is required because the dataset does not contain the complete context needed to make final content decisions.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring

The ranked recommendations should be reviewed periodically as new search and content data becomes available.

Useful monitoring checks include:

- Whether CTR and engagement patterns have changed.
- Whether the distribution of important features has changed substantially.
- Whether the proportion of pages flagged as opportunities changes unexpectedly.
- Whether the highest-ranked pages continue to look reasonable during human review.

### Retrain or review triggers

The model or ranking approach should be reviewed or retrained when:

- New data covers a substantially different period or client mix.
- Performance on newly reviewed pages appears to differ from the original validation results.
- Important input features change substantially in distribution or meaning.
- The ranking produces recommendations that consistently appear unreasonable during human review.
- The business or content-review objective changes.

Retraining should not happen automatically simply because time has passed. New data and the validation process should be reviewed before replacing the existing approach.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [9]:
print("df exists:", "df" in globals())

df exists: True


In [10]:
required_cols = [
    "content_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update"
]

print("Missing columns:", [col for col in required_cols if col not in df.columns])

Missing columns: []


In [11]:
# Create a fresh copy so we don't accidentally change the original df
ranked_queue = df.copy()

# Baseline conditions
visible = ranked_queue["impressions_90d"] >= ranked_queue["impressions_90d"].median()
weak_ctr = ranked_queue["ctr"] <= ranked_queue["ctr"].quantile(0.25)
findable = ranked_queue["avg_position"].between(1, 20)
stale = ranked_queue["days_since_last_update"] >= 180

# Create score
ranked_queue["score"] = 0
ranked_queue.loc[
    visible & weak_ctr & findable,
    "score"
] = 1

ranked_queue.loc[
    visible & weak_ctr & findable & stale,
    "score"
] = 2

# Create reason codes
ranked_queue["reason_code"] = "not_flagged"

ranked_queue.loc[
    visible & weak_ctr & findable,
    "reason_code"
] = "weak_ctr_visible"

ranked_queue.loc[
    visible & weak_ctr & findable & stale,
    "reason_code"
] = "weak_ctr_visible_stale"

print(ranked_queue["reason_code"].value_counts())

reason_code
not_flagged         29200
weak_ctr_visible      800
Name: count, dtype: int64


In [12]:
print(df["days_since_last_update"].describe())

print("\nRows with days_since_last_update >= 180:")
print((df["days_since_last_update"] >= 180).sum())

count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64

Rows with days_since_last_update >= 180:
174


In [13]:
print("Median impressions:", df["impressions_90d"].median())
print("25th percentile CTR:", df["ctr"].quantile(0.25))

print("\nStale pages that are also findable:")
print(
    df.loc[
        (df["days_since_last_update"] >= 180)
        & (df["avg_position"].between(1, 20)),
        [
            "content_id",
            "impressions_90d",
            "ctr",
            "avg_position",
            "days_since_last_update"
        ]
    ].head(20)
)

Median impressions: 731.0
25th percentile CTR: 0.0

Stale pages that are also findable:
                content_id  impressions_90d    ctr  avg_position  \
91    content_48724397d104               38   5.26           2.5   
191   content_4729b57ca036              335   3.28           6.8   
549   content_9433246c4671                3  33.33           4.3   
665   content_e444c00065bd              148   3.38          16.6   
1147  content_ab27c30d81f4              103   0.00           8.9   
1227  content_4f241bad48a3              285   0.00          19.1   
1361  content_74961b456728                1   0.00           4.0   
1659  content_bbca724138f2               75   0.00          12.1   
1735  content_c6a9f1c16dee              180  11.67           3.7   
1778  content_972bb8eb21ec                9  22.22           3.1   
2519  content_0edf498ae135                3   0.00           2.3   
2653  content_1d10143d4e52               20   0.00          13.4   
3190  content_5ccc2e507f1f  

In [14]:
# Show cells/variables currently related to the baseline if available
print("Variables containing 'ctr':")
print([name for name in globals() if "ctr" in name.lower()])

print("\nVariables containing 'baseline':")
print([name for name in globals() if "baseline" in name.lower()])

Variables containing 'ctr':
['weak_ctr']

Variables containing 'baseline':
[]


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.